# 🔍 PS7 — AI-Generated Image Detector
### Neural Nexus Hackathon 2026

**Just run all cells in order — no setup needed!**

- Cell 1 → Installs libraries
- Cell 2 → Downloads model automatically + loads it
- Cell 3 → Upload any image → get prediction
- Cell 4 → Batch test multiple images

**Model:** Hybrid EfficientNet-B0 + ViT-Base/16 | **Accuracy: 98.58%** | **F1: 0.9854** | **AUC: 0.9989**

In [ ]:
# ============================
# CELL 1 — INSTALL
# ============================
!pip install timm gdown -q
print('✅ Libraries ready')

In [ ]:
# ============================
# CELL 2 — AUTO DOWNLOAD MODEL + LOAD
# No manual steps needed — downloads directly!
# ============================
import torch
import torch.nn as nn
import timm
import numpy as np
import gdown
import os
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ---- Auto download model ----
MODEL_PATH = 'model.pth'
FILE_ID    = '1N_4mZUGXTMa_8nbbAJswuUtrLHKxgM_m'

if not os.path.exists(MODEL_PATH):
    print('Downloading model (~364MB)...')
    gdown.download(
        f'https://drive.google.com/uc?id={FILE_ID}',
        MODEL_PATH,
        quiet=False
    )
    print('✅ Download complete!')
else:
    print('✅ Model already downloaded')

# ---- Model Definition ----
class HybridEffNetViT(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.eff = timm.create_model('efficientnet_b0', pretrained=False, num_classes=0)
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Linear(self.eff.num_features + self.vit.num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.classifier(torch.cat([self.eff(x), self.vit(x)], dim=1))

# ---- Config ----
CLASSES  = ['FAKE', 'REAL']
IMG_SIZE = 224
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ---- Load model ----
model = HybridEffNetViT().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print('✅ Model loaded!')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

# ---- Transform ----
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

print('\n🎉 Ready! Run Cell 3 to test any image.')

In [ ]:
# ============================
# CELL 3 — UPLOAD & PREDICT
# Upload any image — real photo or AI-generated
# ============================
from google.colab import files

print('Upload any image (JPG, PNG, WEBP):')
uploaded = files.upload()

for fname in uploaded:
    img = Image.open(fname).convert('RGB')

    tensor = tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), 1).cpu().numpy()[0]

    pred       = int(probs.argmax())
    label      = CLASSES[pred]
    confidence = float(probs.max()) * 100

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    color = 'red' if label == 'FAKE' else 'green'
    ax.set_title(
        f'Prediction: {label}\n'
        f'Confidence: {confidence:.1f}%\n'
        f'FAKE: {probs[0]*100:.1f}%  |  REAL: {probs[1]*100:.1f}%',
        color=color, fontsize=13, fontweight='bold'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print('=' * 40)
    print(f'File      : {fname}')
    print(f'Prediction: {label}')
    print(f'Confidence: {confidence:.1f}%')
    print(f'FAKE prob : {probs[0]*100:.1f}%')
    print(f'REAL prob : {probs[1]*100:.1f}%')
    print('=' * 40)

In [ ]:
# ============================
# CELL 4 — BATCH TEST
# Test multiple images at once
# ============================
from google.colab import files

print('Upload multiple images:')
uploaded = files.upload()

n    = len(uploaded)
cols = min(n, 3)
rows = (n + cols - 1) // cols

fig, axs = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axs = np.array(axs).reshape(rows, cols)

results = []
for i, fname in enumerate(uploaded):
    img    = Image.open(fname).convert('RGB')
    tensor = tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), 1).cpu().numpy()[0]

    pred  = int(probs.argmax())
    label = CLASSES[pred]
    conf  = float(probs.max()) * 100
    results.append({'file': fname, 'label': label, 'conf': conf})

    r, c = divmod(i, cols)
    axs[r][c].imshow(img)
    color = 'red' if label == 'FAKE' else 'green'
    axs[r][c].set_title(f'{label} ({conf:.1f}%)', color=color, fontweight='bold')
    axs[r][c].axis('off')

for j in range(i + 1, rows * cols):
    r, c = divmod(j, cols)
    axs[r][c].axis('off')

plt.suptitle('Batch Prediction Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSummary:')
for r in results:
    print(f"  {r['file']:35s} → {r['label']} ({r['conf']:.1f}%)")